In [23]:
import warnings
import s3fs
import icechunk
import xarray
import hvplot.xarray 
import shutil
import icechunk
import xarray_subset_grid as xsg
from obstore.store import from_url
from obspec_utils.registry import ObjectStoreRegistry
from virtualizarr.parsers import HDFParser
from virtualizarr import open_virtual_dataset

icechunk_fpath = "/home/jovyan/sscofs-icechunk-best"

warnings.filterwarnings("ignore", category=UserWarning)
fs = s3fs.S3FileSystem(anon=True)  
urls = [f"s3://{p}" for p in  fs.glob(f'noaa-nos-ofs-pds/sscofs/netcdf/2026/11/17/*.fields.*.nc')[0:20]]
len(urls)

20

In [2]:

bucket = "s3://noaa-nos-ofs-pds/"
registry = ObjectStoreRegistry({bucket: from_url(bucket, region="us-east-1", skip_signature=True)})
parser = HDFParser(drop_variables=["siglay","siglev","Times"])

In [10]:
%%time

coords = ["time", "lat", "lon"]
vds = open_virtual_dataset(url=urls[0], parser=parser, registry=registry, 
                           loadable_variables=coords)

vds

CPU times: user 542 ms, sys: 79.6 ms, total: 622 ms
Wall time: 4.42 s


<xarray.Dataset> Size: 179MB
Dimensions:             (node: 239734, time: 1, nele: 433410, three: 3,
                         maxnode: 11, maxelem: 9, four: 4, siglay: 10,
                         siglev: 11)
Coordinates:
    lon                 (node) float32 959kB 235.9 235.9 235.9 ... 238.0 238.0
    lat                 (node) float32 959kB 44.39 44.39 44.39 ... 45.63 45.64
  * time                (time) datetime64[ns] 8B 2026-11-17T03:00:00
    lonc                (nele) float32 2MB ManifestArray<shape=(433410,), dty...
    latc                (nele) float32 2MB ManifestArray<shape=(433410,), dty...
Dimensions without coordinates: node, nele, three, maxnode, maxelem, four,
                                siglay, siglev
Data variables: (12/43)
    nprocs              int32 4B ManifestArray<shape=(), dtype=int32, chunks=()>
    partition           (nele) int32 2MB ManifestArray<shape=(433410,), dtype...
    x                   (node) float32 959kB ManifestArray<shape=(239734,), d...
    y                   (node) float32 959kB ManifestArray<shape=(239734,), d...
    xc                  (nele) float32 2MB ManifestArray<shape=(433410,), dty...
    yc                  (nele) float32 2MB ManifestArray<shape=(433410,), dty...
    ...                  ...
    wet_nodes           (time, node) int32 959kB ManifestArray<shape=(1, 2397...
    wet_cells           (time, nele) int32 2MB ManifestArray<shape=(1, 433410...
    wet_nodes_prev_int  (time, node) int32 959kB ManifestArray<shape=(1, 2397...
    wet_cells_prev_int  (time, nele) int32 2MB ManifestArray<shape=(1, 433410...
    wet_cells_prev_ext  (time, nele) int32 2MB ManifestArray<shape=(1, 433410...
    inundation_cells    (time, nele) int32 2MB ManifestArray<shape=(1, 433410...
Attributes: (12/14)
    title:                       SSCOFS
    institution:                 School for Marine Science and Technology
    source:                      FVCOM_4.4.7
    history:                     model started at: 17/11/2026   04:10
    references:                  http://fvcom.smast.umassd.edu, https://githu...
    Conventions:                 CF-1.0
    ...                          ...
    Tidal_Forcing:               Tidal Forcing Time Series Title: sscofs late...
    River_Forcing:               THERE ARE 38 RIVERS IN THIS MODEL.\nRIVER IN...
    GroundWater_Forcing:         GROUND WATER FORCING IS OFF!
    Surface_Heat_Forcing:        FVCOM variable surface heat forcing file:\nF...
    Surface_Wind_Forcing:        FVCOM variable surface Wind forcing:\nFILE N...
    Surface_PrecipEvap_Forcing:  SURFACE PRECIPITATION FORCING IS OFF

In [ ]:
ds_nc= xarray.open_dataset(urls[0],     
                         engine="h5netcdf",
                            storage_options={"anon": True},chunks={})
ds_nc

In [ ]:

bbox =  (-125.3, 44.29, -123.8, 45.11)
ds_sub = ds_nc.xsg.subset_bbox(bbox)
ds_sub

In [20]:
import re
import fsspec
import datetime
def parse_nos_model_run_datestamp_offset(key: str) -> (str, int):
    model_hour, model_date, offset = re.search(r'.t(\d{2})z.(\d{8}).*.[f|n](\d{3})', key).groups()
    model_date = datetime.datetime.strptime(f'{model_date}T{model_hour}', '%Y%m%dT%H') + datetime.timedelta(hours=int(offset))
    model_date_key = model_date.strftime('%Y%m%dT%H')
    return model_date_key, int(offset)

    

glob_expression = '*/sscofs.*.fields.*.nc'
print(urls[0])
prefix, file_type, postfix = re.search(r'(.*).t\d{2}z.\d{8}.(.*).f\d{3}.(.*)', urls[0]).groups()
print(prefix,postfix)
best_time_series_glob = f'{glob_expression}'

fs_read = fsspec.filesystem('s3', anon=True, skip_instance_cache=True)

model_files = fs_read.glob(f's3://noaa-nos-ofs-pds/sscofs/netcdf/2026/11/{best_time_series_glob}')
model_files = sorted(['s3://'+f for f in model_files])

print(len(model_files))

indexes = {}

for f in model_files:
        model_date_key, offset = parse_nos_model_run_datestamp_offset(f)
        if model_date_key not in indexes:
            indexes[model_date_key] = [offset, f]
        else:
            if offset < indexes[model_date_key][0]:
                indexes[model_date_key] = [offset, f]

model_best_files = [x[1] for x in list(indexes.values())]

len(model_best_files)

s3://noaa-nos-ofs-pds/sscofs/netcdf/2026/11/17/sscofs.t03z.20261117.fields.f000.nc
s3://noaa-nos-ofs-pds/sscofs/netcdf/2026/11/17/sscofs nc
5920


511

In [13]:

vds_list = [
    open_virtual_dataset(url=u, parser=parser, registry=registry, loadable_variables=coords)
    for u in model_best_files
]
combined = xarray.concat(vds_list, dim="time", data_vars="minimal", coords="minimal",
                     compat="override", combine_attrs="override")

shutil.rmtree(icechunk_fpath, ignore_errors=True)   # start fresh each run

config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=bucket,
        store=icechunk.s3_store(region="us-east-1", anonymous=True),
    )
)
repo = icechunk.Repository.create(icechunk.local_filesystem_storage(icechunk_fpath), config)
repo.save_config()

# Store the small time coordinate as one chunk, not one chunk per file,
# so the store stays quick to open as more days are added.
combined["time"].encoding["chunks"] = (len(combined.time),)

session = repo.writable_session("main")
combined.vz.to_icechunk(session.store)
session.commit("SSCOFS icechunk")

  2026-11-19T13:50:28.767087Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343

  2026-11-19T13:50:28.775927Z  WARN icechunk_storage::readback: conditional PUT is enabled but `unsafe_use_metadata` is disabled — lost-response recovery for conditional writes requires user metadata to stamp write-ids; without it, transient PUT failures may surface as spurious conflicts even when the write actually landed. See icechunk_storage::Settings::unsafe_use_metadata.
    at icechunk-storage/src/readback.rs:31



'WQRY5PDP1JS0WYYBPEA0'

In [14]:
combined.time

<xarray.DataArray 'time' (time: 511)> Size: 4kB
array(['2026-11-01T03:00:00.000000000', '2026-11-01T03:59:59.999999776',
       '2026-11-01T05:00:00.000000224', ..., '2026-11-22T06:59:59.999999776',
       '2026-11-22T08:00:00.000000224', '2026-11-22T09:00:00.000000000'],
      shape=(511,), dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 4kB 2026-11-01T03:00:00 ... 2026-11-22T09:...
Attributes:
    long_name:  time
    format:     defined reference date
    time_zone:  UTC

In [24]:
creds = icechunk.containers_credentials({bucket: icechunk.s3_credentials(anonymous=True)})
reader = icechunk.Repository.open(icechunk.local_filesystem_storage(icechunk_fpath),
                                  authorize_virtual_chunk_access=creds)
ds_ic = xarray.open_zarr(reader.readonly_session("main").store, consolidated=False, chunks={})
ds_ic = xsg.grids.ugrid.assign_ugrid_topology(ds_ic, face_node_connectivity='nv')
ds_ic

  2026-11-19T14:51:01.881205Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343



<xarray.Dataset> Size: 52GB
Dimensions:             (four: 4, nele: 433410, node: 239734, three: 3,
                         time: 511, maxelem: 9, siglev: 11, maxnode: 11,
                         siglay: 10)
Coordinates:
    lonc                (nele) float32 2MB dask.array<chunksize=(433410,), meta=np.ndarray>
    latc                (nele) float32 2MB dask.array<chunksize=(433410,), meta=np.ndarray>
    lat                 (node) float32 959kB dask.array<chunksize=(239734,), meta=np.ndarray>
    lon                 (node) float32 959kB dask.array<chunksize=(239734,), meta=np.ndarray>
  * time                (time) datetime64[ns] 4kB 2026-11-01T03:00:00 ... 202...
Dimensions without coordinates: four, nele, node, three, maxelem, siglev,
                                maxnode, siglay
Data variables: (12/44)
    a1u                 (four, nele) float32 7MB dask.array<chunksize=(4, 433410), meta=np.ndarray>
    a2u                 (four, nele) float32 7MB dask.array<chunksize=(4, 433410), meta=np.ndarray>
    art2                (node) float32 959kB dask.array<chunksize=(239734,), meta=np.ndarray>
    aw0                 (three, nele) float32 5MB dask.array<chunksize=(3, 433410), meta=np.ndarray>
    awy                 (three, nele) float32 5MB dask.array<chunksize=(3, 433410), meta=np.ndarray>
    iint                (time) int32 2kB dask.array<chunksize=(1,), meta=np.ndarray>
    ...                  ...
    wet_nodes           (time, node) int32 490MB dask.array<chunksize=(1, 239734), meta=np.ndarray>
    x                   (node) float32 959kB dask.array<chunksize=(239734,), meta=np.ndarray>
    yc                  (nele) float32 2MB dask.array<chunksize=(433410,), meta=np.ndarray>
    zeta                (time, node) float32 490MB dask.array<chunksize=(1, 239734), meta=np.ndarray>
    wet_cells_prev_ext  (time, nele) int32 886MB dask.array<chunksize=(1, 433410), meta=np.ndarray>
    mesh                int32 4B 0
Attributes: (12/14)
    title:                       SSCOFS
    institution:                 School for Marine Science and Technology
    source:                      FVCOM_4.4.7
    history:                     model started at: 01/11/2026   04:08
    references:                  http://fvcom.smast.umassd.edu, https://githu...
    Conventions:                 CF-1.0
    ...                          ...
    Tidal_Forcing:               Tidal Forcing Time Series Title: sscofs late...
    River_Forcing:               THERE ARE 38 RIVERS IN THIS MODEL.\nRIVER IN...
    GroundWater_Forcing:         GROUND WATER FORCING IS OFF!
    Surface_Heat_Forcing:        FVCOM variable surface heat forcing file:\nF...
    Surface_Wind_Forcing:        FVCOM variable surface Wind forcing:\nFILE N...
    Surface_PrecipEvap_Forcing:  SURFACE PRECIPITATION FORCING IS OFF

In [19]:
%%time
bbox =  (-125.3, 44.29, -123.8, 45.11)
ds_sub = ds_ic2.xsg.subset_bbox(bbox)
ds_sub

CPU times: user 748 ms, sys: 276 ms, total: 1.02 s
Wall time: 2.23 s


<xarray.Dataset> Size: 358MB
Dimensions:             (four: 4, nele: 2875, node: 1711, time: 511, three: 3,
                         maxnode: 11, siglev: 11, maxelem: 9, siglay: 10)
Coordinates:
    latc                (nele) float32 12kB dask.array<chunksize=(2875,), meta=np.ndarray>
    lonc                (nele) float32 12kB dask.array<chunksize=(2875,), meta=np.ndarray>
    lon                 (node) float32 7kB dask.array<chunksize=(1711,), meta=np.ndarray>
    lat                 (node) float32 7kB dask.array<chunksize=(1711,), meta=np.ndarray>
  * time                (time) datetime64[ns] 4kB 2026-11-01T03:00:00 ... 202...
Dimensions without coordinates: four, nele, node, three, maxnode, siglev,
                                maxelem, siglay
Data variables: (12/44)
    a1u                 (four, nele) float32 46kB dask.array<chunksize=(4, 2875), meta=np.ndarray>
    art1                (node) float32 7kB dask.array<chunksize=(1711,), meta=np.ndarray>
    atmos_press         (time, node) float32 3MB dask.array<chunksize=(1, 1711), meta=np.ndarray>
    awx                 (three, nele) float32 34kB dask.array<chunksize=(3, 2875), meta=np.ndarray>
    h                   (node) float32 7kB dask.array<chunksize=(1711,), meta=np.ndarray>
    a2u                 (four, nele) float32 46kB dask.array<chunksize=(4, 2875), meta=np.ndarray>
    ...                  ...
    yc                  (nele) float32 12kB dask.array<chunksize=(2875,), meta=np.ndarray>
    temp                (time, siglay, node) float32 35MB dask.array<chunksize=(1, 5, 1711), meta=np.ndarray>
    y                   (node) float32 7kB dask.array<chunksize=(1711,), meta=np.ndarray>
    zeta                (time, node) float32 3MB dask.array<chunksize=(1, 1711), meta=np.ndarray>
    wet_nodes           (time, node) int32 3MB dask.array<chunksize=(1, 1711), meta=np.ndarray>
    mesh                int32 4B 0
Attributes: (12/14)
    title:                       SSCOFS
    institution:                 School for Marine Science and Technology
    source:                      FVCOM_4.4.7
    history:                     model started at: 01/11/2026   04:08
    references:                  http://fvcom.smast.umassd.edu, https://githu...
    Conventions:                 CF-1.0
    ...                          ...
    Tidal_Forcing:               Tidal Forcing Time Series Title: sscofs late...
    River_Forcing:               THERE ARE 38 RIVERS IN THIS MODEL.\nRIVER IN...
    GroundWater_Forcing:         GROUND WATER FORCING IS OFF!
    Surface_Heat_Forcing:        FVCOM variable surface heat forcing file:\nF...
    Surface_Wind_Forcing:        FVCOM variable surface Wind forcing:\nFILE N...
    Surface_PrecipEvap_Forcing:  SURFACE PRECIPITATION FORCING IS OFF